⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.9.18"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# `WC_BADGE_DETAILS_D` Batch Load\n",
        "\n",
        "**Source File:** `WC_BADGE_DETAILS_D.sql`\n",
        "**Conversion Date:** 2023-11-20"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from delta.tables import DeltaTable\n",
        "from pyspark.sql import functions as F\n",
        "from pyspark.sql.types import (\n",
        "    StructType, StructField,\n",
        "    StringType, LongType, IntegerType, DoubleType,\n",
        "    DecimalType, TimestampType, DateType, BinaryType, FloatType\n",
        ")\n",
        "from pyspark.sql.window import Window\n",
        "from pyspark.sql import SparkSession\n",
        "\n",
        "spark = SparkSession.builder.getOrCreate()"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "dbutils.widgets.text(\"v_ETL_JOB_TYPE\", \"\")\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"380\") # Hardcoded in ODI script as 380\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")\n",
        "\n",
        "v_etl_job_type    = dbutils.widgets.get(\"v_ETL_JOB_TYPE\")\n",
        "datasource_num_id = int(dbutils.widgets.get(\"DATASOURCE_NUM_ID\"))\n",
        "etl_proc_wid      = int(dbutils.widgets.get(\"ETL_PROC_WID\"))\n",
        "odi_sess_no       = dbutils.widgets.get(\"ODI_SESS_NO\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters\n",
        "\n",
        "Retrieve ETL process parameters such as last extract time and current extract time from `WC_ETL_PARAMETERS` table."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "etl_parameters_df = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_etl_parameters\")\n",
        "    .filter(F.col(\"ETL_JOB_TYPE\") == v_etl_job_type)\n",
        ")\n",
        "\n",
        "etl_last_extract_time = (\n",
        "    etl_parameters_df\n",
        "    .agg(F.max(\"etl_last_extract_time\").alias(\"val\"))\n",
        "    .collect()[0][\"val\"]\n",
        ")\n",
        "\n",
        "etl_current_extract_time = (\n",
        "    etl_parameters_df\n",
        "    .agg(F.max(\"etl_current_extract_time\").alias(\"val\"))\n",
        "    .collect()[0][\"val\"]\n",
        ")\n",
        "\n",
        "etl_row_wid = (\n",
        "    etl_parameters_df\n",
        "    .agg(F.max(\"ROW_WID\").alias(\"val\"))\n",
        "    .collect()[0][\"val\"]\n",
        ")\n",
        "\n",
        "print(f\"ETL Last Extract Time: {etl_last_extract_time}\")\n",
        "print(f\"ETL Current Extract Time: {etl_current_extract_time}\")\n",
        "print(f\"ETL Row WID: {etl_row_wid}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Staging Table: `c_mercury_badge_ts`\n",
        "\n",
        "Drop and recreate the staging table `c_mercury_badge_ts` (formerly `C$_0A7SUCRIPSM1CG2656H955OU5QP`)."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_ts\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "window_spec_dedup = Window.partitionBy(\"ID\").orderBy(F.col(\"INT_INSERT_DATE\").desc(), F.col(\"VERSIONNUMBER\").desc())\n",
        "\n",
        "c_mercury_badge_ts_df = (\n",
        "    spark.table(\"workspace.prxbi_ts.wc_mercury_badge_ts\")\n",
        "    .filter(\n",
        "        (F.col(\"INT_INSERT_DATE\") > F.lit(etl_last_extract_time))\n",
        "        & (F.col(\"INT_INSERT_DATE\") <= F.lit(etl_current_extract_time))\n",
        "    )\n",
        "    .withColumn(\"rn\", F.row_number().over(window_spec_dedup))\n",
        "    .filter(F.col(\"rn\") == 1)\n",
        "    .drop(\"rn\")\n",
        "    .select(\n",
        "        F.col(\"ID\").cast(StringType()).alias(\"ID\"),\n",
        "        F.col(\"BADGELOCATION\").cast(StringType()).alias(\"BADGELOCATION\"),\n",
        "        F.col(\"BADGETOKEN\").cast(StringType()).alias(\"BADGETOKEN\"),\n",
        "        F.col(\"BADGEVERSION\").cast(LongType()).alias(\"BADGEVERSION\"),\n",
        "        F.col(\"CONTACTEMAIL\").cast(StringType()).alias(\"CONTACTEMAIL\"),\n",
        "        F.col(\"CONTACTFIRSTNAME\").cast(StringType()).alias(\"CONTACTFIRSTNAME\"),\n",
        "        F.col(\"CONTACTJOBTITLE\").cast(StringType()).alias(\"CONTACTJOBTITLE\"),\
        "        F.col(\"CONTACTLASTNAME\").cast(StringType()).alias(\"CONTACTLASTNAME\"),\n",
        "        F.col(\"CONTACTPERSONRXMASTERID\").cast(StringType()).alias(\"CONTACTPERSONRXMASTERID\"),\n",
        "        F.col(\"CREATEDBYREGISTRATIONTYPE\").cast(StringType()).alias(\"CREATEDBYREGISTRATIONTYPE\"),\n",
        "        F.col(\"CREATEDBYTYPE\").cast(StringType()).alias(\"CREATEDBYTYPE\"),\n",
        "        F.col(\"CULTURE\").cast(StringType()).alias(\"CULTURE\"),\n",
        "        F.col(\"CUSTOMERTYPE\").cast(StringType()).alias(\"CUSTOMERTYPE\"),\n",
        "        F.col(\"EVENTEDITIONGBSCODE\").cast(StringType()).alias(\"EVENTEDITIONGBSCODE\"),\n",
        "        F.col(\"ISBADGEUPDATE\").cast(StringType()).alias(\"ISBADGEUPDATE\"),\n",
        "        F.col(\"MARKETINGPREFERENCESPROMPTREQUIRED\").cast(StringType()).alias(\"MARKETINGPREFERENCESPROMPTREQU\"),\n",
        "        F.col(\"ORGANISATIONCITY\").cast(StringType()).alias(\"ORGANISATIONCITY\"),\n",
        "        F.col(\"ORGANISATIONCOUNTRYCODE\").cast(StringType()).alias(\"ORGANISATIONCOUNTRYCODE\"),\n",
        "        F.col(\"ORGANISATIONDISPLAYNAME\").cast(StringType()).alias(\"ORGANISATIONDISPLAYNAME\"),\n",
        "        F.col(\"ORGANISATIONRXMASTERID\").cast(StringType()).alias(\"ORGANISATIONRXMASTERID\"),\n",
        "        F.col(\"ORGANISATIONSTATE\").cast(StringType()).alias(\"ORGANISATIONSTATE\"),\n",
        "        F.col(\"PARTICIPATINGORGANISATIONID\").cast(StringType()).alias(\"PARTICIPATINGORGANISATIONID\"),\n",
        "        F.col(\"PRODUCTCODE\").cast(StringType()).alias(\"PRODUCTCODE\"),\n",
        "        F.col(\"QRCODECONTENT\").cast(StringType()).alias(\"QRCODECONTENT\"),\n",
        "        F.col(\"REGISTRATIONID\").cast(StringType()).alias(\"REGISTRATIONID\"),\n",
        "        F.col(\"STATUS\").cast(LongType()).alias(\"STATUS\"),\n",
        "        F.col(\"SUPPORTSTAFFCOMPANYADDRESS\").cast(StringType()).alias(\"SUPPORTSTAFFCOMPANYADDRESS\"),\n",
        "        F.col(\"SUPPORTSTAFFCOMPANYNAME\").cast(StringType()).alias(\"SUPPORTSTAFFCOMPANYNAME\"),\n",
        "        F.col(\"SUPPORTSTAFFMOBILEPHONE\").cast(StringType()).alias(\"SUPPORTSTAFFMOBILEPHONE\"),\n",
        "        F.col(\"SUPPORTSTAFFREPORTSTO\").cast(StringType()).alias(\"SUPPORTSTAFFREPORTSTO\"),\n",
        "        F.col(\"SUPPORTSTAFFSTANDS\").cast(StringType()).alias(\"SUPPORTSTAFFSTANDS\"),\n",
        "        F.col(\"SUPPORTSTAFFUSERACCESS\").cast(StringType()).alias(\"SUPPORTSTAFFUSERACCESS\"),\n",
        "        F.col(\"VERSIONNUMBER\").cast(LongType()).alias(\"VERSIONNUMBER\"),\n",
        "        F.col(\"MOBILEPHONE\").cast(StringType()).alias(\"MOBILEPHONE\"),\n",
        "        F.col(\"FIRSTSCANNEDDATE\").cast(TimestampType()).alias(\"FIRSTSCANNEDDATE\"),\n",
        "        F.col(\"LASTPRINTEDDATE\").cast(TimestampType()).alias(\"LASTPRINTEDDATE\"),\n",
        "        F.col(\"ACCESSVALIDITYMODIFIEDDATE\").cast(TimestampType()).alias(\"ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        F.col(\"CREATEDDATE\").cast(TimestampType()).alias(\"CREATEDDATE\"),\n",
        "        F.col(\"COMPANYPRODUCTCODE\").cast(StringType()).alias(\"COMPANYPRODUCTCODE\"),\n",
        "        F.col(\"PAYMENTSTATUS\").cast(StringType()).alias(\"PAYMENTSTATUS\"),\n",
        "        F.col(\"PHOTOKEY\").cast(StringType()).alias(\"PHOTOKEY\"),\n",
        "        F.col(\"PHOTOSOURCE\").cast(StringType()).alias(\"PHOTOSOURCE\"),\n",
        "        F.col(\"PHOTOSOURCETYPE\").cast(StringType()).alias(\"PHOTOSOURCETYPE\")\n",
        "    )\n",
        ")\n",
        "\n",
        "c_mercury_badge_ts_df.write.format(\"delta\").mode(\"overwrite\").saveAsTable(\"workspace.prxbi_dw.c_mercury_badge_ts\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(f\"Staging count: {spark.table('workspace.prxbi_dw.c_mercury_badge_ts').count()}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table: `i_wc_badge_details_flow`\n",
        "\n",
        "Drop and recreate the flow table `i_wc_badge_details_flow` (formerly `I$_WADHBP95VCNT6J0F17HEGQQJ3GB`).\n",
        "This table joins the staging data with `WC_BADGE_PRODUCT_D` and flags records for insert ('I')."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_flow\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "c_staging_df = spark.table(\"workspace.prxbi_dw.c_mercury_badge_ts\")\n",
        "\n",
        "window_spec_product_dedup = Window.partitionBy(\"SKU\").orderBy(F.col(\"ID\").desc())\n",
        "\n",
        "wc_badge_product_d_df = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_badge_product_d\")\n",
        "    .withColumn(\"COL\", F.rank().over(window_spec_product_dedup))\n",
        "    .filter(F.col(\"COL\") == 1)\n",
        "    .select(\n",
        "        F.col(\"ID\").alias(\"product_id\"),\n",
        "        F.col(\"SKU\").alias(\"product_sku\"),\n",
        "        F.col(\"NAME\").alias(\"PACKAGE_NAME\")\n",
        "    )\n",
        ")\n",
        "\n",
        "i_wc_badge_details_flow_df = (\n",
        "    c_staging_df.alias(\"s\")\n",
        "    .join(\n",
        "        wc_badge_product_d_df.alias(\"p\"),\n",
        "        F.col(\"s.PRODUCTCODE\") == F.col(\"p.product_sku\"),\n",
        "        \"left_outer\"\n",
        "    )\n",
        "    .select(\n",
        "        F.col(\"s.ID\").alias(\"BADGE_ID\"),\n",
        "        F.col(\"s.BADGELOCATION\").alias(\"BADGE_LOCATION\"),\n",
        "        F.col(\"s.BADGETOKEN\").alias(\"BADGE_TOKEN\"),\n",
        "        F.col(\"s.BADGEVERSION\").alias(\"BADGE_VERSION\"),\n",
        "        F.col(\"s.CONTACTEMAIL\").alias(\"CONTACT_EMAIL\"),\n",
        "        F.col(\"s.CONTACTFIRSTNAME\").alias(\"CONTACT_FIRST_NAME\"),\n",
        "        F.col(\"s.CONTACTLASTNAME\").alias(\"CONTACT_LAST_NAME\"),\n",
        "        F.col(\"s.CONTACTJOBTITLE\").alias(\"CONTACT_JOB_TITLE\"),\n",
        "        F.col(\"s.CONTACTPERSONRXMASTERID\").alias(\"CONTACT_PERSON_ID\"),\n",
        "        F.col(\"s.CREATEDBYREGISTRATIONTYPE\").alias(\"CREATION_REG_TYPE\"),\n",
        "        F.col(\"s.CREATEDBYTYPE\").alias(\"CREATION_TYPE\"),\n",
        "        F.col(\"s.CULTURE\").alias(\"CULTURE\"),\n",
        "        F.col(\"s.CUSTOMERTYPE\").alias(\"CUSTOMER_TYPE\"),\n",
        "        F.col(\"s.EVENTEDITIONGBSCODE\").alias(\"EVENT_EDITION_CODE\"),\n",
        "        F.col(\"s.ISBADGEUPDATE\").alias(\"BADGE_UPDATE_FLG\"),\n",
        "        F.col(\"s.MARKETINGPREFERENCESPROMPTREQU\").alias(\"MARKETING_PREF_PROMPT\"),\n",
        "        F.col(\"s.ORGANISATIONDISPLAYNAME\").alias(\"ORG_NAME\"),\n",
        "        F.col(\"s.ORGANISATIONCITY\").alias(\"ORG_CITY\"),\n",
        "        F.col(\"s.ORGANISATIONCOUNTRYCODE\").alias(\"ORG_COUNTRY\"),\n",
        "        F.col(\"s.ORGANISATIONRXMASTERID\").alias(\"ORG_ID\"),\n",
        "        F.col(\"s.ORGANISATIONSTATE\").alias(\"ORG_STATE\"),\n",
        "        F.col(\"s.PARTICIPATINGORGANISATIONID\").alias(\"PARTICIPATING_ORG_ID\"),\n",
        "        F.col(\"s.PRODUCTCODE\").alias(\"PRODUCT_CODE\"),\n",
        "        F.col(\"s.QRCODECONTENT\").alias(\"QR_CODE\"),\n",
        "        F.col(\"s.REGISTRATIONID\").alias(\"REGISTRATION_ID\"),\n",
        "        F.col(\"s.STATUS\").alias(\"STATUS\"),\n",
        "        F.col(\"s.SUPPORTSTAFFCOMPANYNAME\").alias(\"STAFF_COMPANY_NAME\"),\n",
        "        F.col(\"s.SUPPORTSTAFFCOMPANYADDRESS\").alias(\"STAFF_COMPANY_ADDR\"),\n",
        "        F.col(\"s.SUPPORTSTAFFMOBILEPHONE\").alias(\"STAFF_PHONE_NUM\"),\n",
        "        F.col(\"s.SUPPORTSTAFFREPORTSTO\").alias(\"STAFF_REPORTING\"),\n",
        "        F.col(\"s.SUPPORTSTAFFSTANDS\").alias(\"STAFF_STANDS\"),\n",
        "        F.col(\"s.SUPPORTSTAFFUSERACCESS\").alias(\"STAFF_USER_ACCESS\"),\n",
        "        F.col(\"s.VERSIONNUMBER\").alias(\"VERSION_NUM\"),\n",
        "        F.col(\"s.ID\").alias(\"INTEGRATION_ID\"),\n",
        "        F.lit(datasource_num_id).cast(StringType()).alias(\"DATASOURCE_NUM_ID\"),\n",
        "        F.col(\"s.MOBILEPHONE\").alias(\"MOBILEPHONE\"),\n",
        "        F.col(\"s.FIRSTSCANNEDDATE\").alias(\"FIRSTSCANNEDDATE\"),\n",
        "        F.col(\"s.LASTPRINTEDDATE\").alias(\"LASTPRINTEDDATE\"),\n",
        "        F.when(F.col(\"s.FIRSTSCANNEDDATE\").isNotNull(), F.lit(\"Y\")).otherwise(F.lit(\"N\")).alias(\"FIRSTSCANNEDDATE_FLG\"),\n",
        "        F.when(F.col(\"s.LASTPRINTEDDATE\").isNotNull(), F.lit(\"Y\")).otherwise(F.lit(\"N\")).alias(\"LASTPRINTEDDATE_FLG\"),\n",
        "        F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\").alias(\"ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        F.col(\"s.CREATEDDATE\").alias(\"CREATEDDATE\"),\n",
        "        F.col(\"s.COMPANYPRODUCTCODE\").alias(\"COMPANYPRODUCTCODE\"),\n",
        "        F.col(\"s.PAYMENTSTATUS\").alias(\"PAYMENTSTATUS\"),\n",
        "        F.col(\"s.PHOTOKEY\").alias(\"PHOTOKEY\"),\n",
        "        F.col(\"s.PHOTOSOURCE\").alias(\"PHOTOSOURCE\"),\
        "        F.col(\"s.PHOTOSOURCETYPE\").alias(\"PHOTOSOURCETYPE\"),\n",
        "        F.col(\"p.PACKAGE_NAME\").alias(\"PACKAGE_NAME\"),\n",
        "        F.lit(\"I\").cast(StringType()).alias(\"IND_UPDATE\"),\n",
        "        F.current_timestamp().alias(\"W_INSERT_DT\"),\n",
        "        F.current_timestamp().alias(\"W_UPDATE_DT\")\n",
        "    )\n",
        ")\n",
        "\n",
        "i_wc_badge_details_flow_df.write.format(\"delta\").mode(\"overwrite\").saveAsTable(\"workspace.prxbi_dw.i_wc_badge_details_flow\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(f\"Flow count: {spark.table('workspace.prxbi_dw.i_wc_badge_details_flow').count()}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.sql(\"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false\")\n",
        "spark.sql(\"OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_flow ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error / Audit Tables\n",
        "\n",
        "This specific ODI script does not include operations on error tables (`E$_`) or `SNP_CHECK_TAB`."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## PK Violation Detection\n",
        "\n",
        "This specific ODI script does not include explicit PK violation detection and logging into `E$_` tables.\n",
        "Deduplication steps are handled during staging and flow table creation."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Mark Records for Update\n",
        "\n",
        "Identify records in the flow table (`i_wc_badge_details_flow`) that already exist in the target table (`wc_badge_details_d`) and mark them for update ('U')."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "target_table_pk_df = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "    .select(\"INTEGRATION_ID\", \"DATASOURCE_NUM_ID\")\n",
        ")\n",
        "\n",
        "DeltaTable.forName(spark, \"workspace.prxbi_dw.i_wc_badge_details_flow\").alias(\"t\").merge(\n",
        "    target_table_pk_df.alias(\"s\"),\n",
        "    \"t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID\"\n",
        ").whenMatchedUpdate(set={\n",
        "    \"t.IND_UPDATE\": F.lit(\"U\")\n",
        "}).execute()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Merge into Target: `wc_badge_details_d`\n",
        "\n",
        "Merge records from the flow table (`i_wc_badge_details_flow`) into the permanent target table (`wc_badge_details_d`).\n",
        "Records with `IND_UPDATE = 'U'` will update existing rows. Records with `IND_UPDATE = 'I'` will insert new rows.\n",
        "\n",
        "Note: `ROW_WID` is assumed to be an IDENTITY column in the target table DDL and is thus omitted from explicit insert/update clauses."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "flow_df = spark.table(\"workspace.prxbi_dw.i_wc_badge_details_flow\")\n",
        "\n",
        "target_delta_table = DeltaTable.forName(spark, \"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "\n",
        "target_delta_table.alias(\"t\").merge(\n",
        "    flow_df.alias(\"s\"),\n",
        "    \"t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID\"\n",
        ").whenMatchedUpdate(condition=\"s.IND_UPDATE = 'U'\", set={\n",
        "    \"t.BADGE_ID\": F.col(\"s.BADGE_ID\"),\n",
        "    \"t.BADGE_LOCATION\": F.col(\"s.BADGE_LOCATION\"),\n",
        "    \"t.BADGE_TOKEN\": F.col(\"s.BADGE_TOKEN\"),\n",
        "    \"t.BADGE_VERSION\": F.col(\"s.BADGE_VERSION\"),\n",
        "    \"t.CONTACT_EMAIL\": F.col(\"s.CONTACT_EMAIL\"),\n",
        "    \"t.CONTACT_FIRST_NAME\": F.col(\"s.CONTACT_FIRST_NAME\"),\n",
        "    \"t.CONTACT_LAST_NAME\": F.col(\"s.CONTACT_LAST_NAME\"),\n",
        "    \"t.CONTACT_JOB_TITLE\": F.col(\"s.CONTACT_JOB_TITLE\"),\n",
        "    \"t.CONTACT_PERSON_ID\": F.col(\"s.CONTACT_PERSON_ID\"),\n",
        "    \"t.CREATION_REG_TYPE\": F.col(\"s.CREATION_REG_TYPE\"),\n",
        "    \"t.CREATION_TYPE\": F.col(\"s.CREATION_TYPE\"),\n",
        "    \"t.CULTURE\": F.col(\"s.CULTURE\"),\n",
        "    \"t.CUSTOMER_TYPE\": F.col(\"s.CUSTOMER_TYPE\"),\n",
        "    \"t.EVENT_EDITION_CODE\": F.col(\"s.EVENT_EDITION_CODE\"),\n",
        "    \"t.BADGE_UPDATE_FLG\": F.col(\"s.BADGE_UPDATE_FLG\"),\n",
        "    \"t.MARKETING_PREF_PROMPT\": F.col(\"s.MARKETING_PREF_PROMPT\"),\n",
        "    \"t.ORG_NAME\": F.col(\"s.ORG_NAME\"),\n",
        "    \"t.ORG_CITY\": F.col(\"s.ORG_CITY\"),\n",
        "    \"t.ORG_COUNTRY\": F.col(\"s.ORG_COUNTRY\"),\n",
        "    \"t.ORG_ID\": F.col(\"s.ORG_ID\"),\n",
        "    \"t.ORG_STATE\": F.col(\"s.ORG_STATE\"),\n",
        "    \"t.PARTICIPATING_ORG_ID\": F.col(\"s.PARTICIPATING_ORG_ID\"),\n",
        "    \"t.PRODUCT_CODE\": F.col(\"s.PRODUCT_CODE\"),\n",
        "    \"t.QR_CODE\": F.col(\"s.QR_CODE\"),\n",
        "    \"t.REGISTRATION_ID\": F.col(\"s.REGISTRATION_ID\"),\n",
        "    \"t.STATUS\": F.col(\"s.STATUS\"),\n",
        "    \"t.STAFF_COMPANY_NAME\": F.col(\"s.STAFF_COMPANY_NAME\"),\n",
        "    \"t.STAFF_COMPANY_ADDR\": F.col(\"s.STAFF_COMPANY_ADDR\"),\n",
        "    \"t.STAFF_PHONE_NUM\": F.col(\"s.STAFF_PHONE_NUM\"),\n",
        "    \"t.STAFF_REPORTING\": F.col(\"s.STAFF_REPORTING\"),\n",
        "    \"t.STAFF_STANDS\": F.col(\"s.STAFF_STANDS\"),\n",
        "    \"t.STAFF_USER_ACCESS\": F.col(\"s.STAFF_USER_ACCESS\"),\n",
        "    \"t.VERSION_NUM\": F.col(\"s.VERSION_NUM\"),\n",
        "    \"t.MOBILEPHONE\": F.col(\"s.MOBILEPHONE\"),\n",
        "    \"t.FIRSTSCANNEDDATE\": F.col(\"s.FIRSTSCANNEDDATE\"),\n",
        "    \"t.LASTPRINTEDDATE\": F.col(\"s.LASTPRINTEDDATE\"),\n",
        "    \"t.FIRSTSCANNEDDATE_FLG\": F.col(\"s.FIRSTSCANNEDDATE_FLG\"),\n",
        "    \"t.LASTPRINTEDDATE_FLG\": F.col(\"s.LASTPRINTEDDATE_FLG\"),\n",
        "    \"t.ACCESSVALIDITYMODIFIEDDATE\": F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "    \"t.CREATEDDATE\": F.col(\"s.CREATEDDATE\"),\n",
        "    \"t.COMPANYPRODUCTCODE\": F.col(\"s.COMPANYPRODUCTCODE\"),\n",
        "    \"t.PAYMENTSTATUS\": F.col(\"s.PAYMENTSTATUS\"),\n",
        "    \"t.PHOTOKEY\": F.col(\"s.PHOTOKEY\"),\n",
        "    \"t.PHOTOSOURCE\": F.col(\"s.PHOTOSOURCE\"),\n",
        "    \"t.PHOTOSOURCETYPE\": F.col(\"s.PHOTOSOURCETYPE\"),\n",
        "    \"t.PACKAGE_NAME\": F.col(\"s.PACKAGE_NAME\"),\n",
        "    \"t.W_UPDATE_DT\": F.current_timestamp()\n",
        "}).whenNotMatchedInsert(condition=\"s.IND_UPDATE = 'I'\", values={\n",
        "    \"BADGE_ID\": F.col(\"s.BADGE_ID\"),\n",
        "    \"BADGE_LOCATION\": F.col(\"s.BADGE_LOCATION\"),\n",
        "    \"BADGE_TOKEN\": F.col(\"s.BADGE_TOKEN\"),\n",
        "    \"BADGE_VERSION\": F.col(\"s.BADGE_VERSION\"),\n",
        "    \"CONTACT_EMAIL\": F.col(\"s.CONTACT_EMAIL\"),\n",
        "    \"CONTACT_FIRST_NAME\": F.col(\"s.CONTACT_FIRST_NAME\"),\n",
        "    \"CONTACT_LAST_NAME\": F.col(\"s.CONTACT_LAST_NAME\"),\n",
        "    \"CONTACT_JOB_TITLE\": F.col(\"s.CONTACT_JOB_TITLE\"),\n",
        "    \"CONTACT_PERSON_ID\": F.col(\"s.CONTACT_PERSON_ID\"),\n",
        "    \"CREATION_REG_TYPE\": F.col(\"s.CREATION_REG_TYPE\"),\n",
        "    \"CREATION_TYPE\": F.col(\"s.CREATION_TYPE\"),\n",
        "    \"CULTURE\": F.col(\"s.CULTURE\"),\n",
        "    \"CUSTOMER_TYPE\": F.col(\"s.CUSTOMER_TYPE\"),\n",
        "    \"EVENT_EDITION_CODE\": F.col(\"s.EVENT_EDITION_CODE\"),\n",
        "    \"BADGE_UPDATE_FLG\": F.col(\"s.BADGE_UPDATE_FLG\"),\n",
        "    \"MARKETING_PREF_PROMPT\": F.col(\"s.MARKETING_PREF_PROMPT\"),\n",
        "    \"ORG_NAME\": F.col(\"s.ORG_NAME\"),\n",
        "    \"ORG_CITY\": F.col(\"s.ORG_CITY\"),\n",
        "    \"ORG_COUNTRY\": F.col(\"s.ORG_COUNTRY\"),\n",
        "    \"ORG_ID\": F.col(\"s.ORG_ID\"),\n",
        "    \"ORG_STATE\": F.col(\"s.ORG_STATE\"),\n",
        "    \"PARTICIPATING_ORG_ID\": F.col(\"s.PARTICIPATING_ORG_ID\"),\n",
        "    \"PRODUCT_CODE\": F.col(\"s.PRODUCT_CODE\"),\n",
        "    \"QR_CODE\": F.col(\"s.QR_CODE\"),\n",
        "    \"REGISTRATION_ID\": F.col(\"s.REGISTRATION_ID\"),\n",
        "    \"STATUS\": F.col(\"s.STATUS\"),\n",
        "    \"STAFF_COMPANY_NAME\": F.col(\"s.STAFF_COMPANY_NAME\"),\n",
        "    \"STAFF_COMPANY_ADDR\": F.col(\"s.STAFF_COMPANY_ADDR\"),\n",
        "    \"STAFF_PHONE_NUM\": F.col(\"s.STAFF_PHONE_NUM\"),\n",
        "    \"STAFF_REPORTING\": F.col(\"s.STAFF_REPORTING\"),\n",
        "    \"STAFF_STANDS\": F.col(\"s.STAFF_STANDS\"),\n",
        "    \"STAFF_USER_ACCESS\": F.col(\"s.STAFF_USER_ACCESS\"),\n",
        "    \"VERSION_NUM\": F.col(\"s.VERSION_NUM\"),\n",
        "    \"INTEGRATION_ID\": F.col(\"s.INTEGRATION_ID\"),\n",
        "    \"DATASOURCE_NUM_ID\": F.col(\"s.DATASOURCE_NUM_ID\"),\n",
        "    \"MOBILEPHONE\": F.col(\"s.MOBILEPHONE\"),\n",
        "    \"FIRSTSCANNEDDATE\": F.col(\"s.FIRSTSCANNEDDATE\"),\n",
        "    \"LASTPRINTEDDATE\": F.col(\"s.LASTPRINTEDDATE\"),\n",
        "    \"FIRSTSCANNEDDATE_FLG\": F.col(\"s.FIRSTSCANNEDDATE_FLG\"),\n",
        "    \"LASTPRINTEDDATE_FLG\": F.col(\"s.LASTPRINTEDDATE_FLG\"),\n",
        "    \"ACCESSVALIDITYMODIFIEDDATE\": F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "    \"CREATEDDATE\": F.col(\"s.CREATEDDATE\"),\n",
        "    \"COMPANYPRODUCTCODE\": F.col(\"s.COMPANYPRODUCTCODE\"),\n",
        "    \"PAYMENTSTATUS\": F.col(\"s.PAYMENTSTATUS\"),\n",
        "    \"PHOTOKEY\": F.col(\"s.PHOTOKEY\"),\n",
        "    \"PHOTOSOURCE\": F.col(\"s.PHOTOSOURCE\"),\n",
        "    \"PHOTOSOURCETYPE\": F.col(\"s.PHOTOSOURCETYPE\"),\n",
        "    \"PACKAGE_NAME\": F.col(\"s.PACKAGE_NAME\"),\n",
        "    \"W_INSERT_DT\": F.current_timestamp(),\n",
        "    \"W_UPDATE_DT\": F.current_timestamp()\n",
        "}).execute()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Optimize Target\n",
        "\n",
        "Optimize the target Delta table for query performance using ZORDER."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.sql(\"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false\")\n",
        "spark.sql(\"OPTIMIZE workspace.prxbi_dw.wc_badge_details_d ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Cleanup\n",
        "\n",
        "Drop temporary staging and flow tables."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_flow\")\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_ts\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Validation\n",
        "\n",
        "Display the total count of the target table and a sample of the data."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "target_count = spark.table(\"workspace.prxbi_dw.wc_badge_details_d\").count()\n",
        "print(f\"Final target table count: {target_count}\")\n",
        "\n",
        "print(\"Sample data from target table:\")\n",
        "display(spark.table(\"workspace.prxbi_dw.wc_badge_details_d\").limit(10))"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "spark.stop()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Conversion Notes and Manual Actions Required\n",
        "\n",
        "1.  **Schema Configuration**: Ensure `workspace.prxbi_dw` and `workspace.prxbi_ts` schemas exist and are correctly configured in your Databricks environment.\n",
        "2.  **`WC_ETL_PARAMETERS` Table**: The source script queries `WC_ETL_PARAMETERS` for `etl_last_extract_time`, `etl_current_extract_time`, and `ROW_WID` (`etl_proc_wid`). This table is assumed to exist in `workspace.prxbi_dw` and contain the necessary parameters, which should be updated by a preceding ETL control job.\n",
        "3.  **`DATASOURCE_NUM_ID`**: The ODI script hardcodes `DATASOURCE_NUM_ID` to `380` when inserting into the flow table. This has been replicated, but if this value should be dynamic, the `DATASOURCE_NUM_ID` widget can be used to override it or further logic can be implemented.\n",
        "4.  **`ROW_WID` (Target Table IDENTITY Column)**: The target table `WC_BADGE_DETAILS_D` in the ODI script uses `WC_BADGE_DETAILS_D_SEQ.NEXTVAL` for `ROW_WID` during inserts. In Databricks, `ROW_WID` is assumed to be a `GENERATED ALWAYS AS IDENTITY` column in the target table's DDL (Delta Lake feature). If this is not the case, and `ROW_WID` must be populated manually, you would need to implement a mechanism for generating unique, sequential IDs, such as `F.monotonically_increasing_id()` or a custom UDF, ensuring uniqueness and appropriate range.\n",
        "5.  **`NOLOGGING` and `/*+ APPEND */`**: These Oracle-specific hints are removed as they are not applicable in Databricks Delta Lake, which handles transaction logging and append optimization automatically.\n",
        "6.  **`DBMS_STATS.GATHER_TABLE_STATS`**: Oracle statistics gathering calls have been removed. Delta Lake automatically collects statistics. For temporary tables (C$ and I$), `OPTIMIZE` commands for ZORDER are generally not required unless specifically for performance of downstream steps during the session, which is not typical for transient staging tables. An `OPTIMIZE` command with ZORDER was added for the flow table, and also for the final target table, with the `spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled` flag set to false as required by the prompt, to ensure it executes without issues related to missing column statistics."
      ]
    }
  ]
}